In [21]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import (
    gaussian_kde,
    probplot,
    boxcox,
    zscore,
    kstest,
    shapiro
)
from sklearn.preprocessing import PowerTransformer
from statsmodels.stats.diagnostic import lilliefors

plt.style.use("seaborn-v0_8-whitegrid")

DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =====================================================
# DATASET CONFIGURATION
# =====================================================
DATASET_CONFIGS = {
    "insurance": {
        "file_path": "data/insurance.csv",
        "column_name": "charges",
        "separator": ",",
        "category_column": None,
        "category_value": None
    },

    # =================================================
    # IRIS
    # =================================================
    "iris_all": {
        "file_path": "data/iris.csv",
        "column_name": "sepal_length",
        "separator": ",",
        "category_column": None,
        "category_value": None
    },

    "iris_setosa": {
        "file_path": "data/iris.csv",
        "column_name": "sepal_length",
        "separator": ",",
        "category_column": "species",
        "category_value": "setosa"
    },

    "iris_versicolor": {
        "file_path": "data/iris.csv",
        "column_name": "sepal_length",
        "separator": ",",
        "category_column": "species",
        "category_value": "versicolor"
    },

    "iris_virginica": {
        "file_path": "data/iris.csv",
        "column_name": "sepal_length",
        "separator": ",",
        "category_column": "species",
        "category_value": "virginica"
    },

    # =================================================
    # WINE RED
    # =================================================
    "wine_red_alcohol": {
        "file_path": "data/winequality-red.csv",
        "column_name": "alcohol",
        "separator": ";",
        "category_column": None,
        "category_value": None
    },

    "wine_red_quality_high": {
        "file_path": "data/winequality-red.csv",
        "column_name": "alcohol",
        "separator": ";",
        "category_column": "quality",
        "category_value": 7
    },

    "wine_red_quality_low": {
        "file_path": "data/winequality-red.csv",
        "column_name": "alcohol",
        "separator": ";",
        "category_column": "quality",
        "category_value": 5
    },

    # =================================================
    # WINE WHITE
    # =================================================
    "wine_white_alcohol": {
        "file_path": "data/winequality-white.csv",
        "column_name": "alcohol",
        "separator": ";",
        "category_column": None,
        "category_value": None
    },

    "wine_white_quality_high": {
        "file_path": "data/winequality-white.csv",
        "column_name": "alcohol",
        "separator": ";",
        "category_column": "quality",
        "category_value": 7
    },

    "wine_white_quality_low": {
        "file_path": "data/winequality-white.csv",
        "column_name": "alcohol",
        "separator": ";",
        "category_column": "quality",
        "category_value": 5
    }
}


# =====================================================
# SETTINGS
# =====================================================
ALPHA = 0.05
SAVE_GRID_AND_NO_GRID = True
SAVE_TRANSFORMED_DATASET = True

# Available methods:
# none, zscore, log, boxcox, yeojohnson, all
TRANSFORMATION_METHOD = "all"

# Choose one key from DATASET_CONFIGS
SELECTED_DATASET_KEY = "iris_setosa"


# =====================================================
# COLORS
# =====================================================
COLOR_HIST = "#9ecae1"
COLOR_DENSITY = "#d62728"
COLOR_QQ_POINTS = "#1f77b4"
COLOR_QQ_LINE = "#ff7f0e"


# =====================================================
# SAVE FIGURE
# =====================================================
def save_figure(fig, name):
    plt.tight_layout(rect=[0, 0, 1, 0.97])

    if SAVE_GRID_AND_NO_GRID:
        for ax in fig.axes:
            ax.grid(True, alpha=0.28)
        fig.savefig(OUTPUT_DIR / f"{name}_grid.svg", format="svg")

        for ax in fig.axes:
            ax.grid(False)
        fig.savefig(OUTPUT_DIR / f"{name}.svg", format="svg")
    else:
        fig.savefig(OUTPUT_DIR / f"{name}.svg", format="svg")

    plt.close(fig)


# =====================================================
# LOAD DATA
# =====================================================
def load_column(
    file_path,
    numeric_column,
    sep=",",
    category_column=None,
    category_value=None
):
    file_path = Path(file_path)
    if not file_path.exists():
        file_path = DATA_DIR / file_path.name

    if not file_path.exists():
        raise FileNotFoundError(f"Dataset file not found: {file_path}")

    df = pd.read_csv(file_path, sep=sep)

    if category_column is not None and category_value is not None:
        df = df[df[category_column] == category_value].copy()

    data = pd.to_numeric(df[numeric_column], errors="coerce").dropna()

    return df, data


# =====================================================
# TRANSFORMATIONS
# =====================================================
def apply_transformation(data, method):
    data = pd.Series(data).dropna().astype(float)

    if method == "none":
        return data, None

    if method == "zscore":
        transformed = zscore(data)
        return pd.Series(transformed).dropna(), None

    if method == "log":
        transformed = np.log1p(data)
        return pd.Series(transformed).dropna(), None

    if method == "boxcox":
        if (data <= 0).any():
            raise ValueError("Box-Cox requires strictly positive values.")
        transformed, lam = boxcox(data)
        return pd.Series(transformed).dropna(), lam

    if method == "yeojohnson":
        pt = PowerTransformer(method="yeo-johnson", standardize=False)
        transformed = pt.fit_transform(data.to_numpy().reshape(-1, 1)).flatten()
        return pd.Series(transformed).dropna(), None

    raise ValueError("Invalid transformation method")


def get_methods(user_method):
    if user_method == "all":
        return ["none", "zscore", "log", "boxcox", "yeojohnson"]
    if user_method == "none":
        return ["none"]
    return ["none", user_method]


# =====================================================
# NORMALITY TESTS
# =====================================================
def normality_tests(data):
    data = np.asarray(data, dtype=float)
    data = data[~np.isnan(data)]

    # Standardize internally for KS / Lilliefors comparison to N(0,1)
    data_std = zscore(data)

    ks_stat, ks_p = kstest(data_std, "norm")
    lillie_stat, lillie_p = lilliefors(data_std, dist="norm")
    shapiro_stat, shapiro_p = shapiro(data)

    return {
        "KS-Test": {"stat": ks_stat, "p": ks_p},
        "Lilliefors": {"stat": lillie_stat, "p": lillie_p},
        "Shapiro-Wilk": {"stat": shapiro_stat, "p": shapiro_p}
    }


# =====================================================
# SAFE NAMES
# =====================================================
def make_safe_name(text):
    return (
        str(text)
        .replace(" ", "_")
        .replace("(", "")
        .replace(")", "")
        .replace("/", "_")
        .replace("-", "_")
    )


# =====================================================
# PLOT DASHBOARD
# =====================================================
def plot_dashboard(data, methods, title, file_name, alpha=0.05):
    rows = len(methods)

    fig, axes = plt.subplots(rows, 3, figsize=(18, rows * 5))

    if rows == 1:
        axes = np.array([axes])

    for row, method in enumerate(methods):
        transformed, lam = apply_transformation(data, method)

        ax1 = axes[row, 0]
        ax2 = axes[row, 1]
        ax3 = axes[row, 2]

        method_label = method.upper()
        if lam is not None:
            method_label += f" (λ={lam:.3f})"

        # ---------------------------------------------
        # 1. Histogram + Density
        # ---------------------------------------------
        ax1.hist(
            transformed,
            bins=30,
            density=True,
            alpha=0.75,
            color=COLOR_HIST,
            edgecolor="white",
            linewidth=0.8
        )

        x_grid = np.linspace(np.min(transformed), np.max(transformed), 500)
        kde = gaussian_kde(transformed)

        ax1.plot(
            x_grid,
            kde(x_grid),
            color=COLOR_DENSITY,
            linewidth=2.5,
            label="Density line"
        )

        ax1.set_title(
            f"{method_label} - Histogram + Density",
            fontsize=11,
            fontweight="bold"
        )
        ax1.set_xlabel("Value")
        ax1.set_ylabel("Density")
        ax1.legend()

        # ---------------------------------------------
        # 2. QQ Plot
        # ---------------------------------------------
        (osm, osr), (slope, intercept, r) = probplot(transformed, dist="norm")

        ax2.scatter(
            osm,
            osr,
            s=20,
            alpha=0.8,
            color=COLOR_QQ_POINTS
        )

        ax2.plot(
            osm,
            slope * osm + intercept,
            color=COLOR_QQ_LINE,
            linewidth=2,
            label="Reference line"
        )

        ax2.set_title(
            f"{method_label} - QQ Plot",
            fontsize=11,
            fontweight="bold"
        )
        ax2.set_xlabel("Theoretical Quantiles")
        ax2.set_ylabel("Sample Quantiles")
        ax2.legend()

        # ---------------------------------------------
        # 3. Normality Tests
        # ---------------------------------------------
        ax3.axis("off")
        results = normality_tests(transformed)

        rows_table = []
        for test_name, values in results.items():
            stat = values["stat"]
            pval = values["p"]

            if pval >= alpha:
                decision = "Normal-like"
            else:
                decision = "Not normal"

            rows_table.append([
                test_name,
                f"{stat:.4f}",
                f"{pval:.4f}",
                decision
            ])

        table = ax3.table(
            cellText=rows_table,
            colLabels=["Test", "Statistic", "P-value", "Decision"],
            cellLoc="center",
            loc="center"
        )
        table.auto_set_font_size(False)
        table.set_fontsize(9)
        table.scale(1.1, 1.7)

        ax3.set_title(
            f"{method_label} - Normality Tests",
            fontsize=11,
            fontweight="bold",
            pad=12
        )

    fig.suptitle(title, fontsize=16, fontweight="bold")
    save_figure(fig, file_name)


# =====================================================
# SAVE TRANSFORMED DATASET
# =====================================================
def add_transformed_columns(
    df,
    original_column,
    methods,
    output_file
):
    df_copy = df.copy()

    original_numeric = pd.to_numeric(df_copy[original_column], errors="coerce")
    numeric_mask = original_numeric.notna()
    original_clean = original_numeric[numeric_mask]

    for method in methods:
        transformed_data, _ = apply_transformation(original_clean, method)

        new_column_name = f"{make_safe_name(original_column)}_{method}"
        df_copy[new_column_name] = np.nan
        df_copy.loc[numeric_mask, new_column_name] = transformed_data.values

    df_copy.to_csv(output_file, index=False)
    print("Saved:", output_file)


# =====================================================
# MAIN
# =====================================================
config = DATASET_CONFIGS[SELECTED_DATASET_KEY]

file_path = config["file_path"]
column_name = config["column_name"]
separator = config["separator"]
category_column = config["category_column"]
category_value = config["category_value"]

df, original_data = load_column(
    file_path=file_path,
    numeric_column=column_name,
    sep=separator,
    category_column=category_column,
    category_value=category_value
)

methods = get_methods(TRANSFORMATION_METHOD)

file_prefix = os.path.splitext(os.path.basename(file_path))[0]
safe_column = make_safe_name(column_name)

chart_file_name = f"{SELECTED_DATASET_KEY}_{file_prefix}_{safe_column}_{TRANSFORMATION_METHOD}"

title = f"{SELECTED_DATASET_KEY} | {column_name}"
if category_column is not None and category_value is not None:
    title += f" | {category_column} = {category_value}"

plot_dashboard(
    data=original_data,
    methods=methods,
    title=title,
    file_name=chart_file_name,
    alpha=ALPHA
)

if SAVE_TRANSFORMED_DATASET:
    output_dataset_file = OUTPUT_DIR / f"{SELECTED_DATASET_KEY}_{file_prefix}_with_transformed_columns.csv"
    add_transformed_columns(
        df=df,
        original_column=column_name,
        methods=methods,
        output_file=output_dataset_file
    )

Saved: output\iris_setosa_iris_with_transformed_columns.csv
